# 04 模型评估

仅评估本项目真实训练产生的权重，不回退到官方预训练权重。

In [ ]:
from pathlib import Path
from time import perf_counter
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

DATA_YAML = '../data/data.yaml'
SAMPLE_IMAGE = Path('../data/evaluation_sample.jpg')
WEIGHTS = {name: Path(f'../weights/{name}/best.pt') for name in ['YOLO11n', 'YOLO11s', 'YOLO26n', 'YOLO26s']}
missing = [str(path) for path in WEIGHTS.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('找不到真实训练权重：' + ', '.join(missing))

In [ ]:
rows = []
for name, weight in WEIGHTS.items():
    model = YOLO(str(weight))
    metrics = model.val(data=DATA_YAML, split='test')
    started = perf_counter()
    model.predict(source=str(SAMPLE_IMAGE), verbose=False)
    inference_ms = (perf_counter() - started) * 1000
    rows.append({'model': name, 'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr), 'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map), 'weight_mb': weight.stat().st_size / 1024 / 1024, 'inference_ms': inference_ms})
comparison = pd.DataFrame(rows).sort_values(['map50_95', 'inference_ms'], ascending=[False, True])
comparison['recommended'] = comparison['model'].eq(comparison.iloc[0]['model'])
Path('../reports').mkdir(exist_ok=True)
comparison.to_csv('../reports/model_comparison.csv', index=False, encoding='utf-8-sig')
comparison

In [ ]:
ax = comparison.plot.scatter(x='inference_ms', y='map50_95', title='模型精度与速度对比')
for _, row in comparison.iterrows():
    ax.annotate(row['model'], (row['inference_ms'], row['map50_95']))
ax.set_xlabel('单张图片推理耗时（ms）')
ax.set_ylabel('mAP50-95')
plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=160)